In [15]:
import time
import numpy as np

from pid.optimizers.tilde_pid import exact_tilde_union_info_minimizer
from pid.optimizers.thin_pid import exact_thin_pid_minimizer
from pid.utils import generate_randomBC

In [16]:
step=100
start=100
max_dX=1000
num_runs=1
ratio=[0.6, 0.1]

In [17]:
# Generate dx, dy, dT sequence
R = np.array([1, ratio[0]])
sequence = np.arange(start, max_dX + 1, step)
XY_dim = np.array([seq * R for seq in sequence]).astype(int)  # TXY_dim = sequence * R = [dT, dx, dy]
print(XY_dim)

[[ 100  100]
 [ 200  200]
 [ 300  300]
 [ 400  400]
 [ 500  500]
 [ 600  600]
 [ 700  700]
 [ 800  800]
 [ 900  900]
 [1000 1000]]


In [18]:
X_dim = np.arange(start, max_dX + 1, step)
Y_dim = np.ones(len(sequence), dtype=int) * 100
print(X_dim, Y_dim)

[ 100  200  300  400  500  600  700  800  900 1000] [100 100 100 100 100 100 100 100 100 100]


In [19]:
dtype = [('dT', int), ('dx', int), ('dy', int), ('obj_tilde', float), ('obj_thin', float),
             ('execution_time_tilde', float), ('execution_time_thin', float), ('itr_tilde', int),
             ('itr_thin', int)]
result_comparison = np.zeros((len(sequence), num_runs), dtype=dtype)  # compare tilde and thin pid

In [21]:
for i, seq in enumerate(sequence):
    dx, dy = XY_dim[i]
    # dx, dy = X_dim[i], Y_dim[i]
    dT = 2
    print(f"\nRunning config [dx, dy, dT]: {dx:3d}-{dy:3d}-{dT:3d}")

    for j in range(num_runs):

        hx, hy = generate_randomBC(dT, dx, dy)
        if dx + dy < 1000:  # only run the original algorithm when the size is small, otherwise it takes too long
            start_time1 = time.time()
            # sig_tilde, obj_tilde, itr_tilde, _ = exact_tilde_union_info_minimizer(hx, hy, ret_obj=True, reg=1e-8)
            end_time1 = time.time()
        else:
            sig_tilde, obj_tilde, itr_tilde = 0, 0, 0
            end_time1, start_time1 = 0.0, 0.0
            # print(f"Skipping tilde pid for {dx:3d}-{dy:3d}-{dT:3d}")

        start_time2 = time.time()
        sig_thin, obj_thin, itr_thin, _ = exact_thin_pid_minimizer(hx, hy, ret_obj=True, reg=1e-8)
        end_time2 = time.time()

        execution_time_tilde = end_time1 - start_time1
        execution_time_thin = end_time2 - start_time2
        result_comparison[i, j] = (
            dT, dx, dy, obj_tilde, obj_thin, execution_time_tilde, execution_time_thin, itr_tilde, itr_thin
        )

        # print(f"obj_tilde: {obj_tilde:4.9f}, obj_thin: {obj_thin:4.9f}, time_tilde: {execution_time_tilde:3.4f}, time_thin: {execution_time_thin:3.4f}")

    # print average results
    print(f"Average obj_tilde: {np.mean(result_comparison[i]['obj_tilde']):4.9f}, obj_thin: {np.mean(result_comparison[i]['obj_thin']):4.9f}, time_tilde: {np.mean(result_comparison[i]['execution_time_tilde']):3.4f}, time_thin: {np.mean(result_comparison[i]['execution_time_thin']):3.4f}")


Running config [dx, dy, dT]: 100-100-  2
Average obj_tilde: 6.763247778, obj_thin: 6.676583766, time_tilde: 0.0000, time_thin: 0.2189

Running config [dx, dy, dT]: 200-200-  2
Average obj_tilde: 6.763247778, obj_thin: 7.825789455, time_tilde: 0.0000, time_thin: 1.8207

Running config [dx, dy, dT]: 300-300-  2
Average obj_tilde: 6.763247778, obj_thin: 6.326020242, time_tilde: 0.0000, time_thin: 12.6490

Running config [dx, dy, dT]: 400-400-  2
Average obj_tilde: 6.763247778, obj_thin: 8.710241191, time_tilde: 0.0000, time_thin: 6.7520

Running config [dx, dy, dT]: 500-500-  2
Average obj_tilde: 0.000000000, obj_thin: 8.917420241, time_tilde: 0.0000, time_thin: 31.7660

Running config [dx, dy, dT]: 600-600-  2
Average obj_tilde: 0.000000000, obj_thin: 9.248555105, time_tilde: 0.0000, time_thin: 34.8514

Running config [dx, dy, dT]: 700-700-  2
Average obj_tilde: 0.000000000, obj_thin: 9.555998988, time_tilde: 0.0000, time_thin: 21.8177

Running config [dx, dy, dT]: 800-800-  2
Average o